# 14 — V3-E2: MiniLM embedding cache audit

This notebook checks the operational evidence produced in V3-E2. It is
read-only: it does not recompute embeddings, train classifiers, select a
threshold, or access `test`, `stress`, or `monitor`.

Its purpose is to confirm that the local cache exactly matches the authorized
development keys and that its files remain intact. Outputs are aggregate-only;
no narrative or individual embedding is published.


## Frozen scientific contract

V3-E0 fixed the `debt_credit_management` critical class, frozen MiniLM with
LinearSVC as the primary approach, word/character TF-IDF with LinearSVC as the
control, every unique critical positive, negative ratios 1:5, 1:10, and 1:15,
and expanding temporal validation with natural prevalence and a 0.40 precision
floor.

E2 implements only the frozen MiniLM representation. Predictive comparisons
across representations, ratios, and folds belong to V3-E3 and do not exist at
this point.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v3.reporting import (
    audit_v3_e2_cache,
    render_v3_e2_audit,
)


## Complete cache audit

The call below validates the manifest and protocol hash; hashes and sizes for
all 19 shard pairs; array format and dimensions; key schema; group uniqueness;
and exact agreement with `train` and `validation` in the scientific cache. It
also computes aggregate-only L2-norm diagnostics.


In [ ]:
audit = audit_v3_e2_cache(PROJECT_ROOT)
print(render_v3_e2_audit(audit, 'en-US'))

V3-E2 — EMBEDDING CACHE AUDIT

status: VALID_COMPLETE_DEVELOPMENT_CACHE
model: sentence-transformers/all-MiniLM-L6-v2
revision: bc57282bc374d33e0d6c4de27f12dc1c2a87f37a
shape: 591,532 x 384
dtype: float16

INTEGRITY
  shards: 19
  unique_groups: 591,532
  train_rows: 345,552
  validation_rows: 245,980
  missing_keys: 0
  extra_keys: 0
  raw_narratives_persisted: False
  forbidden_partitions_accessed: False
  cache_size_mib: 444.94
  ordered_key_sha256: 87F159E3BD1461E82F03FAF7D5AAAA966B0E7F12E5D45B4827476012CAF25AB4
  manifest_sha256: 09C67DF7B0EF6BC25FCFFFAB3E4EEEFB37E3EE75E2F0B07052F44AB9CEA235E5

EXECUTION
  elapsed_seconds: 717.5706
  rows_per_second: 824.35
  batch_size: 128
  device: NVIDIA GeForce RTX 3050 Laptop GPU
  peak_reserved_mib: 488.00

EMBEDDING L2 NORMS (DIAGNOSTIC)
  train: n=345,552 | min=0.9995 | p01=0.9996 | median=1.0000 | mean=1.0000 | p99=1.0004 | max=1.0005 | zeros=0
  validation: n=245,980 | min=0.9995 | p01=0.9996 | median=1.0000 | mean=1.0000 | p99=1.0004 |

## Interpretation

A valid status establishes cache integrity, coverage, and operational
feasibility. L2 norms help detect zero vectors, non-finite vectors, or coarse
scale drift; they do not measure separability or predictive quality.

The observed norms are approximately 1 because the pinned checkpoint itself
contains an internal `2_Normalize` module. The frozen
`normalize_embeddings=False` setting means the `encode` call does not request
an additional normalization step; it does not remove the checkpoint module.

This notebook therefore cannot establish whether MiniLM is better or worse
than TF-IDF. That question starts in V3-E3, using the approved rolling-origin
folds, frozen sampling, and temporal calibration.
